In [1]:
import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

In [2]:
!pip install -r requirements.txt

In [3]:
import os
os.environ['NUMEXPR_MAX_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '2'
import numexpr as ne

In [4]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
import openai
openai.api_key = userdata.get('OPENAI_API_KEY')

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from llama_index.embeddings.langchain import LangchainEmbedding

embed_model = LangchainEmbedding(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, load_index_from_storage, ServiceContext
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1)
Settings.embed_model = embed_model


#try:
 #   storage_context = StorageContext.from_defaults(persist_dir='./storage/cache/andrew/sleep')
  #  index = load_index_from_storage(storage_context)
   # print('loading from disk')
#except:
documents = SimpleDirectoryReader('assets/AndrewHuberman/sleep').load_data()
index = VectorStoreIndex.from_documents(documents)
index.storage_context.persist(persist_dir='./storage/cache/andrew/sleep/')


In [8]:
query_engine = index.as_query_engine()
response = query_engine.query("How does sun light affect sleep?")
print(response)

Sunlight affects sleep by helping to regulate the circadian clock, which is the internal system that determines sleep-wake cycles. Exposure to bright light, particularly in the morning, can enhance the ability to fall asleep and improve overall sleep quality. This is because sunlight provides essential cues that help synchronize the circadian system to the day-night cycle. Additionally, even on cloudy days, being outside in natural light can still be beneficial, as the intensity of outdoor light is generally greater than that of indoor lighting. The duration of exposure to light is also important; spending more time outside can provide additional cues for the circadian system, further supporting better sleep patterns.
